# Fine-tune Mask2Former on Pascal VOC and collect metrics

Fine-tune a [Mask2Former](https://huggingface.co/docs/transformers/en/model_doc/mask2former) model for
semantic segmentation on the Pascal VOC 2012 tables created in **`create-pascal-voc-semseg-table`**, and
collect per-sample predictions and IoU back into a 3LC `Run`.

<!-- Tags: ["semantic-segmentation", "pascal-voc", "mask2former", "hugging-face", "training"] -->

Mask2Former is a *universal* segmentation model — it predicts a set of binary masks plus a class for
each. We use it in **semantic mode**: starting from an ADE20k-semantic checkpoint, we swap in a 21-class
head for VOC and fine-tune, then collapse the mask set back to a dense per-pixel label map with the
processor's `post_process_semantic_segmentation`. The void boundary (id 255) is excluded from both the
loss and the metrics. Every few epochs we collect, per sample: the predicted segmentation (as RLE), mean
IoU, the per-image confusion matrix (via the core metrics helper), and a pooled decoder embedding
(reduced to 2D with PaCMAP after training).

## Project setup

In [ ]:
# Parameters
PROJECT_NAME = "3LC Tutorials - Pascal VOC 2012"
DATASET_NAME = "pascal-voc-2012"
RUN_NAME = "Fine-tune Mask2Former"
RUN_DESCRIPTION = "Mask2Former (semantic mode) fine-tuned on Pascal VOC 2012"
CHECKPOINT = "facebook/mask2former-swin-tiny-ade-semantic"
EPOCHS = 10
BATCH_SIZE = 4
LR = 5e-5
IMAGE_SIZE = 384
COLLECT_FREQUENCY = 5   # collect per-sample metrics every N epochs (and on the final epoch)
NUM_WORKERS = 0         # 0 is safest in notebooks: notebook-defined transforms can't be pickled to
                        # spawned DataLoader workers. Bump it only when running as an importable module.
SEED = 42
INSTALL_DEPENDENCIES = True

## Install dependencies

In [ ]:
if INSTALL_DEPENDENCIES:
    %pip install -q 3lc transformers torch torchvision tqdm pacmap

## Imports

In [ ]:
from __future__ import annotations

import random

import numpy as np
import tlc
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

from tlc.integration.torch.samplers import create_sampler
from tlc.sample_types import (
    SemanticSegmentation,
    semseg_classes,
)
from tlc.schemas import SemanticSegmentationRleSchema
from tlc.helpers.semantic_segmentation_metrics import semantic_segmentation_metrics

## Class universe

VOC has 21 classes (background + 20 objects); `void` (id 255) is the ignore boundary. `background` (id 0)
is not a value-map class — it rides in the column schema's metadata — so both the GT and predicted value
maps show the 20 objects, with the GT map additionally carrying the tagged `void` class. `id2label` /
`label2id` are what we hand to the model's reinitialized classification head (the model still predicts
background as id 0).

In [ ]:
VOC_CLASS_NAMES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car",
    "cat", "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]
BACKGROUND_ID = 0
VOID_ID = 255
NUM_CLASSES = len(VOC_CLASS_NAMES)  # 21

GT_CLASSES = semseg_classes(
    {**{i: n for i, n in enumerate(VOC_CLASS_NAMES)}, VOID_ID: "void"}, background=BACKGROUND_ID, void=VOID_ID
)
PREDICTED_CLASSES = semseg_classes({i: n for i, n in enumerate(VOC_CLASS_NAMES)}, background=BACKGROUND_ID)

id2label = {i: n for i, n in enumerate(VOC_CLASS_NAMES)}
label2id = {n: i for i, n in enumerate(VOC_CLASS_NAMES)}

## Load the processor and model

The processor turns images (and, for training, segmentation maps) into the mask-classification targets
Mask2Former expects, and post-processes predictions back into dense label maps. We set `ignore_index` to
the VOC void id and keep `do_reduce_labels=False` (VOC's background is a real class, not an ignore label).

The model loads from an ADE20k-semantic checkpoint; `num_labels=21` with `ignore_mismatched_sizes=True`
reinitializes the classification head for VOC while keeping the pretrained backbone and pixel decoder.

In [ ]:
processor = AutoImageProcessor.from_pretrained(
    CHECKPOINT,
    size={"height": IMAGE_SIZE, "width": IMAGE_SIZE},
    ignore_index=VOID_ID,
    do_reduce_labels=False,
)

model = Mask2FormerForUniversalSegmentation.from_pretrained(
    CHECKPOINT,
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

## Sample transform

We feed the table to the DataLoader via `Table.with_transform` rather than a custom `Dataset`. The
transform returns a `(PIL image, dense label map)` pair per row; the Mask2Former processor handles
resizing and target construction in the collate function. Curation is handled by the sampler
(`create_sampler` drops zero-weight rows), so Dashboard exclusions take effect on the next run.

In [ ]:
def voc_sample_transform(sample: dict) -> tuple:
    seg: SemanticSegmentation = sample["mask"]
    return sample["image"].convert("RGB"), seg.mask.astype(np.int64)


def collate_fn(batch: list) -> dict:
    images = [image for image, _ in batch]
    masks = [mask for _, mask in batch]
    return processor(images=images, segmentation_maps=masks, return_tensors="pt")

## Metrics collection

For each sample we predict at the original image size, then derive the dense label map with
`post_process_semantic_segmentation`. We store the predicted segmentation (as RLE), the mean IoU and the
per-image confusion matrix (both from the core helper, void excluded), and a pooled embedding from the
transformer decoder's per-query hidden states.

In [ ]:
def collect_metrics(run: tlc.Run, table: tlc.Table, model, device: torch.device, epoch: int) -> float:
    predictions: list[np.ndarray] = []
    ious: list[float] = []
    confusion_matrices: list[list[int]] = []
    embeddings: list[np.ndarray] = []

    model.eval()
    with torch.no_grad():
        for idx in tqdm(range(len(table)), desc="collect", leave=False):
            row = table[idx]
            image = row["image"].convert("RGB")
            seg: SemanticSegmentation = row["mask"]
            width, height = image.size

            inputs = processor(images=image, return_tensors="pt").to(device)
            outputs = model(**inputs)

            pred_map = (
                processor.post_process_semantic_segmentation(outputs, target_sizes=[(height, width)])[0]
                .cpu()
                .numpy()
                .astype(np.int32)
            )

            # Pooled decoder embedding: mean over the object queries -> one vector per image.
            decoder_hidden = outputs.transformer_decoder_last_hidden_state  # (1, num_queries, hidden)
            embeddings.append(decoder_hidden.mean(dim=1).squeeze(0).cpu().numpy().astype(np.float32))

            # Bare label map; the metrics writer serializes it via the predicted_segmentation
            # schema below (which records background id 0 in its metadata), so no wrapper is needed.
            predictions.append(pred_map)

            m = semantic_segmentation_metrics(
                pred_map, seg.mask, GT_CLASSES, background=BACKGROUND_ID, include_background=True
            )
            ious.append(m["mean_iou"])
            confusion_matrices.append([int(x) for cm_row in m["confusion_matrix"] for x in cm_row])

    run.add_metrics(
        {
            "predicted_segmentation": predictions,
            "iou": ious,
            "confusion_matrix": confusion_matrices,
            "embedding": embeddings,
            "epoch": [epoch] * len(ious),
        },
        schema={
            # Background (id 0) rides in the schema metadata, not the value map.
            "predicted_segmentation": SemanticSegmentationRleSchema(classes=PREDICTED_CLASSES, background=BACKGROUND_ID),
            "embedding": tlc.schemas.EmbeddingSchema(shape=len(embeddings[0])),
        },
        foreign_table_url=table.url,
    )
    # Average only over images with a defined IoU; a degenerate image yields nan, and a nan headline
    # would be logged into the run object as an invalid JSON token.
    finite = [v for v in ious if np.isfinite(v)]
    return float(np.mean(finite)) if finite else float("nan")

## Reproducibility helpers

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Load the tables and initialize the Run

`.latest()` picks up the newest revision of each table, so retraining consumes any Dashboard curation
without code changes.

In [ ]:
seed_everything(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")
model = model.to(device)

train_table = tlc.Table.from_names(table_name="train", dataset_name=DATASET_NAME, project_name=PROJECT_NAME).latest()
val_table = tlc.Table.from_names(table_name="val", dataset_name=DATASET_NAME, project_name=PROJECT_NAME).latest()

train_view = train_table.with_transform(voc_sample_transform)
train_sampler = create_sampler(train_table, weighted=False, shuffle=True)
print(f"Train: {len(train_sampler)} of {len(train_table)} rows after weight filtering | Val: {len(val_table)}")

run = tlc.init(
    PROJECT_NAME,
    run_name=RUN_NAME,
    description=RUN_DESCRIPTION,
    parameters={"checkpoint": CHECKPOINT, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "image_size": IMAGE_SIZE},
)

train_loader = DataLoader(
    train_view, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_WORKERS, collate_fn=collate_fn,
)

## Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    n_batches = 0
    for batch in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
        optimizer.zero_grad()
        outputs = model(
            pixel_values=batch["pixel_values"].to(device),
            mask_labels=[m.to(device) for m in batch["mask_labels"]],
            class_labels=[c.to(device) for c in batch["class_labels"]],
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        n_batches += 1
    train_loss /= max(n_batches, 1)

    log_entry = {"epoch": epoch, "train_loss": train_loss}

    is_final = epoch == EPOCHS - 1
    if (epoch + 1) % COLLECT_FREQUENCY == 0 or is_final:
        val_miou = collect_metrics(run, val_table, model, device, epoch=epoch)
        log_entry["val_miou"] = val_miou

    tlc.log(log_entry)
    print("  ".join(f"{k}={v:.4f}" if k != "epoch" else f"epoch {v}" for k, v in log_entry.items()))

## Collect on the train split and reduce embeddings

We collect once more on the train split (so both splits have final-epoch metrics), then fit one PaCMAP
model and apply it across every metrics table so they share a single 2D space.

In [ ]:
collect_metrics(run, train_table, model, device, epoch=EPOCHS - 1)

print("Reducing embeddings (PaCMAP)...")
run.reduce_embeddings_by_foreign_table_url(val_table.url, method="pacmap", delete_source_tables=True)

run.set_status_completed()
print(f"Run: {run.url}")

## Next steps

Open the `Run` in the [3LC Dashboard](https://dashboard.3lc.ai) to overlay predictions on the images,
sort by IoU to find the hardest classes and images, and use the per-image confusion matrix to see which
VOC classes get confused. Exclude or relabel samples in the Dashboard, then re-run — `.latest()` picks up
the curated revision automatically.